# Fine-tune Cross-Encoder CV-JD v0.7

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.6 — batch sinh mới hoàn toàn bằng `scripts/generate_synthetic_data.py` sau khi sửa generator (water-filling label balancing + skill-overlap authoring theo từng cặp + decision-priority rule cho poor_match).

| | |
|---|---|
| **Dataset** | v0.6 — 2,520 train / 540 validation / 540 test (3,600 pairs, cân bằng tuyệt đối 20%/class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` regression (label = score / 100) |
| **Evaluator** | Spearman correlation + LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.7` |

**Mục tiêu**: xác nhận baseline trên dataset v0.6 — dataset nhỏ hơn v0.4 (7k) và v0.5 (13.35k) đáng kể, nhưng lần đầu tiên có class balance tuyệt đối (không lệch qua weak_match/excellent_match như các batch trước) và có poor_match thật (không phải 0% như v0.4/v0.5). So sánh với ceiling cũ:
- v0.2: 60.76% (7k, MSELoss)
- v0.5/v0.6 (data v0.5): 60.95% (13.35k, MSELoss + Spearman)

Nếu run 1 tiệm cận hoặc vượt ~60%, đó là tín hiệu tốt để tiếp tục sinh thêm data (hướng tới 7k+) cho lần chạy chính thức.


In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.7
!git pull origin experiment/cross-encoder-v0.7


In [ ]:
!pip install -r requirements.txt


In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.6/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Debug run — sanity check (1 epoch, 40 samples)


In [ ]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.6/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20


## Full training — 10 epochs, save to Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.7", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --data-dir datasets/versions/v0.6/cross_encoder \
    --loss mse \
    --evaluator spearman \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.7 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.7_report.json \
    --epochs 10 \
    --batch-size 16


In [ ]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.7_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))


In [ ]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.7_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.7_report.json",
)
print(f"Saved to {reports_dir}")
